In [0]:
ric = 'kic'

In [0]:
%sql

-- 동일 유저 당 월별 1개 로그(가장 마지막 값)만 추출 요청드립니다.
with filter_cc as (
    select distinct country_code from kic_data_dimension.common.country_code where region='KIC'
)
, filter_nl as (
    select  'NL_HC_DEVICELIST' as tb_nm
            , date_ym 
            , mac_addr 
            , nl.X_User_Number 
            , nl.X_Device_Country 
            , nl.normal_log:dvc_list
            , nl.log_create_time
    from   kic_data_ods.tlamp.normal_log_webos25 nl 
    inner join filter_cc cc on nl.X_Device_Country = cc.country_code
    where  nl.context_name = 'com.webos.app.homeconnect'
    and  nl.message_id = 'NL_HC_DEVICELIST'
    and  nl.date_ym = '2026-02' 
)
 
select tb_nm, date_ym, mac_addr, X_User_Number, X_Device_Country, dvc_list
from (
    select *,
           row_number() over (partition by X_User_Number, date_ym order by log_create_time desc) as rn
    from filter_nl
) t
where rn = 1

In [0]:



    with filter_cc as (
        select distinct country_code from kic_data_dimension.common.country_code where region='{ric.upper()}'
    )# 로그 전체 추출 요청드립니다.
df = spark.sql(f'''
    
    with filter_cc as (
        select distinct country_code from kic_data_dimension.common.country_code where region='{ric.upper()}'
    )

    select 'NL_HC_CONTROL' as tb_nm
           , date_ym 
           , mac_addr 
           , nl.X_User_Number 
           , nl.X_Device_Country 
           , nl.normal_log:dvc_control.dvcType as dvcType 
           , nl.normal_log:dvc_control.name as name
           , nl.normal_log:dvc_control.control as control
           , nl.normal_log:dvc_control.controlValue as controlValue
    from   {ric}_data_ods.tlamp.normal_log_webos25 nl
    inner join filter_cc cc 
    on nl.X_Device_Country = cc.country_code
    where  nl.context_name = 'com.webos.app.homeconnect'
    and  nl.message_id = 'NL_HC_CONTROL'
    and  nl.date_ym > '2026-02' 
    and nl.date_ym < '2026-05'
''')

df.coalesce(1).write.format("com.databricks.spark.csv").option("header", "true").mode("overwrite")\
    .save(f's3://s3-lge-he-inbound-{ric}-dev/HEDS/HEDS-1393')